# 00 - macOS Apple Silicon MPS preflight

Run this notebook before downloading Granite. It verifies the real machine, Python environment, MPS allocation, and a small forward/backward operation. The target machine for this path has 64 GB unified memory.

This notebook does not download a model or start training.

In [ ]:
from __future__ import annotations

import importlib.metadata
import platform
import subprocess
import sys

import torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("PyTorch:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

if platform.system() != "Darwin" or platform.machine() not in {"arm64", "aarch64"}:
    raise RuntimeError("This notebook expects macOS on Apple Silicon")
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable; reinstall a native arm64 PyTorch environment")

## Inspect the Mac

Confirm that the reported memory is 64 GB and that the Python process is native arm64 rather than an Intel process running through Rosetta.

In [ ]:
hardware = subprocess.run(
    ["system_profiler", "SPHardwareDataType"],
    check=True,
    capture_output=True,
    text=True,
).stdout
print(hardware)

## Exercise MPS for real

Availability flags alone are insufficient. This cell allocates tensors, performs forward and backward operations, synchronizes the device, and reads the result back on the CPU.

In [ ]:
device = torch.device("mps")
x = torch.randn(512, 512, device=device, requires_grad=True)
w = torch.randn(512, 512, device=device, requires_grad=True)
loss = (x @ w).square().mean()
loss.backward()
torch.mps.synchronize()
print("MPS loss:", float(loss.detach().cpu()))
print("Gradient finite:", bool(torch.isfinite(x.grad).all().cpu()))
del x, w, loss
torch.mps.empty_cache()

## Verify package versions

Install the environments from the repository root before continuing:

```bash
uv sync --extra train --extra notebook
uv run python -m ipykernel install --user --name granite-switch-adapter-guide
uv run jupyter lab
```


In [ ]:
packages = ["accelerate", "datasets", "peft", "torch", "transformers"]
for package in packages:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = "NOT INSTALLED"
    print(f"{package}: {version}")

## Acceptance gate

Continue only when:

- The machine is Apple Silicon.
- MPS is built and available.
- The real MPS forward/backward operation succeeds.
- The process reports the expected 64 GB unified memory.
- All training packages are installed.
